In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import pitch_name_to_pitch, pitch_to_frequency

In [ ]:
def karplus_strong(pitch: str, duration: float, f_s: int = 44100) -> pq.Audio:
    """A plucked string: a burst of noise through a damped, low-passed feedback delay line."""
    f = pitch_to_frequency(pitch_name_to_pitch(pitch))
    M = round(f_s / f - 0.5)                         # the average adds half a sample of delay
    g = 10 ** (-3 * (M + 0.5) / (f_s * duration))    # decay by 60 dB over the duration
    N = int(duration * f_s)
    x = np.zeros(N)
    x[:M] = np.random.uniform(-1.0, 1.0, M)          # excitation: one period of white noise
    y = np.zeros(N)
    for n in range(N):
        y[n] = x[n]
        if n > M:                                    # delay, low-pass, and damping
            y[n] += g * 0.5 * (y[n - M] + y[n - M - 1])
    return pq.Audio(y, f_s)


# Strum an E major chord on a six-string guitar, one string every 40 ms
strum = pq.Score([(0.04 * i, {"pitch": pitch, "duration": 3.0})
                  for i, pitch in enumerate(["E2", "B2", "E3", "G#3", "B3", "E4"])])
pq.play(strum.render(karplus_strong), normalize=True)